# 深度解析: `Pump`水工建筑物

**相关模块:** `preissmann_model/structures.py`

## 目的
本教程旨在深入解析`Pump`（水泵）这一水工建筑物的物理原理和在模型中的实现方式。水泵通常用于将水从低处抽到高处，是水资源调度和排涝工程中的关键设备。

此笔记本将：
1.  解释`Pump`模型所基于的物理关系——**水泵特性曲线（Pump Characteristic Curve）**。
2.  详细说明`Pump`类的参数的物理意义。
3.  通过数值实验，验证模型计算出的扬程是否与理论公式一致。
4.  绘制水泵的“流量-扬程”关系曲线。

## 1. 物理原理: 水泵特性曲线

水泵的性能由其特性曲线描述，该曲线定义了水泵能够提供的扬程（`ΔH`，即抬升的水头高度）与通过水泵的流量（`Q`）之间的关系。在模型中，这个关系被简化为一个二次多项式：

$$ \Delta H = aQ^2 + bQ + c $$

其中：
- `ΔH`: 扬程 (m), 等于 `Z_down - Z_up`。
- `Q`: 通过水泵的流量 (m³/s)。
- `a, b, c`: 特性曲线的系数。通常，`a`是一个负数，表示随着流量增大，扬程会下降；`c`是关死扬程（zero-flow head），即流量为0时能提供的最大扬程。

与`Gate`类似，在Preissmann求解器中，这个方程也被线性化以用于求解。

## 2. 环境设置和参数定义

我们先定义一个`Pump`实例所需的参数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.structures import Pump

# --- 水泵参数 ---
# curve_coeffs: 特性曲线的系数 (a, b, c)
pump_params = {
    'name': 'test_pump',
    'node_index': 1, 
    'curve_coeffs': (-0.05, 0.1, 8.0) # H = -0.05*Q^2 + 0.1*Q + 8
}

pump = Pump(**pump_params)
print("Pump对象创建成功。")

## 3. 数值实验与理论验证

为了验证`Pump`类的实现，我们可以直接根据其特性曲线公式计算不同流量下的理论扬程，并绘制出这条曲线。

In [ ]:
def calculate_theoretical_head(pump, q):
    a, b, c = pump.curve_coeffs
    return a * q**2 + b * q + c

# --- 实验: 改变流量，观察扬程变化 ---
q_values = np.linspace(0, 12, 50)
head_theoretical = [calculate_theoretical_head(pump, q) for q in q_values]

plt.figure(figsize=(8, 6))
plt.plot(q_values, head_theoretical, 'r-s', label='Pump Characteristic Curve')
plt.title(f'Pump Performance Curve (H = {pump.curve_coeffs[0]}Q² + {pump.curve_coeffs[1]}Q + {pump.curve_coeffs[2]})')
plt.xlabel('Discharge (Q) [m³/s]')
plt.ylabel('Head (ΔH) [m]')
plt.grid(True)
plt.legend()
plt.show()

## 4. 在模型中验证

我们也可以在一个简单的模型中进行验证。我们设置一个上游水位`Z_up`，然后给定一个流量`Q`，让模型通过`get_linearized_equation`反解出下游水位`Z_down`。`Z_down - Z_up`应该就等于我们上面计算出的理论扬程。

从`get_linearized_equation`的`rhs`项 `-( (Z_down - Z_up) - (aQ^2 + bQ + c) )` 可知，当系统收敛时，`rhs`趋近于0，即 `Z_down - Z_up = aQ^2 + bQ + c`。

In [ ]:
Z_up = 2.0
Q_test = 10.0

# 理论计算
theoretical_delta_h = calculate_theoretical_head(pump, Q_test)
theoretical_z_down = Z_up + theoretical_delta_h

# 从模型方程反算 (当rhs=0时)
a, b, c = pump.curve_coeffs
Z_down_from_eq = Z_up + (a * Q_test**2 + b * Q_test + c)

print(f"给定 Q = {Q_test} m³/s, Z_up = {Z_up} m:")
print(f"  - 理论计算出的扬程 ΔH: {theoretical_delta_h:.3f} m")
print(f"  - 理论计算出的下游水位 Z_down: {theoretical_z_down:.3f} m")
print(f"  - 从模型方程反算出的下游水位 Z_down: {Z_down_from_eq:.3f} m")

assert np.isclose(theoretical_z_down, Z_down_from_eq), "模型实现与理论不符!"
print("\n验证成功，模型实现与理论公式一致。")

### 结论

通过本教程，我们：
- 理解了水泵在模型中是如何通过其“流量-扬程”特性曲线来表达的。
- 验证了`Pump`类的实现与理论公式是匹配的。

这为在更复杂的管网或河网模型中可靠地使用水泵组件奠定了基础。